# Extended Kalman Filter, A Revisit on the Non-Linear State Space Model

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp.kalman_1d_ekf import (
    kalman_filter_1d_ekf,
    kalman_dk_smoother_1d_ekf,
)
from wunkui.models.ssp import a_to_lam, plot_states

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = False 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)

Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
# def make_fourier_series(t, period, order):
#     """
#     Args
#     ----
#     t: array-like, time points at which to evaluate the Fourier series
#     period: int, the period of the seasonality
#     order: int, the number of Fourier terms to include
#     """
#     sin_terms = jnp.array([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
#     cos_terms = jnp.array([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)])
#     return jnp.concatenate((sin_terms, cos_terms), axis=0).transpose(1, 0)

# x_glb_trend = jnp.arange(len(y)) / 365.25
# x_annual_seas = make_fourier_series(jnp.arange(len(y)), period=365.25, order=3)
# x_weekly_seas = make_fourier_series(jnp.arange(len(y)), period=7, order=2)
# x_seas = jnp.concatenate((x_annual_seas, x_weekly_seas), axis=1)
# print(x_seas.shape)

In [ ]:
def disclose_at(
    n_steps: int,
    n_states: int,
    true_state: jnp.ndarray,
    regressors: list,
    positivity_mask: jnp.ndarray,
    n_periods: int = 3,
    n_points: int = 7,
    seed: int = 42,
    obs_scale: float = 1e-3,
) -> tuple:
    """Construct a_obs and P_obs by disclosing the ground-truth latent
    state over n_periods randomly drawn windows of n_points consecutive steps.

    Parameters
    ----------
    n_steps : int
        Total number of time steps.
    n_states : int
        Number of latent states (intercept + regressors).
    true_state : jnp.ndarray, shape (n_states,)
        Ground-truth state vector in observation space (λ for channels, linear
        for intercept). Channel values are converted to log-intensity a-space
        via a = 2·log(λ) before disclosure.
    regressors : list[str]
        Regressor names; determines which state indices get a finite variance.
    positivity_mask : jnp.ndarray, shape (n_states,), dtype bool
        True for channel states (log-intensity space), False for intercept.
    n_periods : int
        Number of disclosure windows to draw.
    n_points : int
        Number of consecutive steps per window.
    seed : int
        RNG seed for reproducibility.
    obs_scale : float
        Standard deviation in a-space expressing confidence in the disclosed
        state. For channels, relates to λ-space uncertainty via σ_a ≈ 2·σ_λ/λ
        (delta method). Smaller → tighter prior; larger → more diffuse.

    Returns
    -------
    a_obs : jnp.ndarray, shape (n_steps, n_states)
        Disclosed state means in a-space; zero where not disclosed.
    P_obs : jnp.ndarray, shape (n_steps, n_states)
        Disclosed state variances; inf where no information is provided
        (intercept column always inf, undisclosed timesteps always inf).
    obs_idx : np.ndarray
        Sorted array of all disclosed time indices (for plotting).
    """
    rng = np.random.default_rng(seed)
    # sample n_periods window starts; ensure each window fits within n_steps
    starts = rng.choice(n_steps - n_points + 1, size=n_periods, replace=False)
    obs_idx = np.unique(np.concatenate([np.arange(s, s + n_points) for s in starts]))

    # convert true λ → a for channel states: a = 2·log(λ)
    # intercept stays in linear space as-is
    true_state_a = jnp.where(
        positivity_mask,
        2.0 * jnp.log(jnp.clip(true_state, 1e-8, None)),
        true_state,
    )

    a_obs = jnp.zeros((n_steps, n_states))
    # default inf = zero precision = no information; pure filter carries through
    P_obs = jnp.full((n_steps, n_states), jnp.inf)

    a_obs = a_obs.at[obs_idx].set(true_state_a)
    # intercept has no ground truth so its variance stays inf; channels get obs_scale
    var_row = jnp.where(positivity_mask, obs_scale ** 2, jnp.inf)
    P_obs = P_obs.at[obs_idx].set(var_row)

    return a_obs, P_obs, obs_idx

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
ssp_priors = {}

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    true_state = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        a_obs, P_obs, obs_idx = disclose_at(
            n_steps=n_steps,
            n_states=n_states,
            true_state=true_state,
            regressors=regressors,
            positivity_mask=positivity_idx,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        ssp_priors.update({"a_obs": a_obs, "P_obs": P_obs, "obs_idx": obs_idx})
        print("n disclosed steps:", len(obs_idx))
    else:
        ssp_priors.update({"a_obs": None, "P_obs": None, "obs_idx": np.array([], dtype=int)})
        print("disclosure disabled")

    print("a_obs:", ssp_priors["a_obs"] if ssp_priors["a_obs"] is None else ssp_priors["a_obs"].shape)
    print("P_obs:", ssp_priors["P_obs"] if ssp_priors["P_obs"] is None else ssp_priors["P_obs"].shape)
else:
    ssp_priors.update({"a_obs": None, "P_obs": None, "obs_idx": np.array([], dtype=int)})
    print("no ground truth state available; disclosure disabled")

In [ ]:
INIT_LAMBDA = 0.1
ssp_priors["a0"] = jnp.concatenate([
    jnp.zeros(1),
    jnp.full(n_states - 1, 2.0 * jnp.log(INIT_LAMBDA)),
])
ssp_priors["P0"] = jnp.ones(n_states)
ssp_priors["sigma_q_loc_prior"] = np.array([0.05, 0.01])
ssp_priors["sigma_q_scale_prior"] = np.array([0.05, 0.01])

# unpack for downstream use
a_obs               = ssp_priors["a_obs"]
P_obs               = ssp_priors["P_obs"]
obs_idx             = ssp_priors["obs_idx"]
a0                  = ssp_priors["a0"]
P0                  = ssp_priors["P0"]
sigma_q_loc_prior   = ssp_priors["sigma_q_loc_prior"]
sigma_q_scale_prior = ssp_priors["sigma_q_scale_prior"]

positivity_mask = positivity_idx.astype(bool)

print("a0 shape:", a0.shape)
print(f"  a0[0] = {a0[0]:.3f}  (intercept, linear space)")
print(f"  a0[1] = {a0[1]:.3f}  (channel, log-intensity → λ₀ = exp(0.5·a0) = {jnp.exp(0.5 * a0[1]):.3f})")
print("P0:", P0)

In [ ]:
# Test run — mixed EKF: intercept linear, channels exp(0.5·a)
sigma_h = jnp.array(0.1)
sigma_q_raw = jnp.array([0.01, 0.01])  # [level, shared_regressors]
sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
print("sigma_q expanded:", sigma_q.shape, sigma_q)

lp, at, Pt, _, _, _ = kalman_filter_1d_ekf(
    a0=a0,
    P0=P0,
    sigma_h=sigma_h,
    sigma_q=sigma_q,
    y=y,
    Z=Z,
    logp=True,
    positivity_idx=positivity_mask,
)

# recover intensities: intercept stays as-is, channels via exp(0.5·a)
lam = jnp.where(positivity_mask, jnp.exp(jnp.clip(0.5 * at, -10.0, 10.0)), at)

print("lp:", lp)
print("at shape:", at.shape)
print("intercept at[-1,0]:", at[-1, 0], "  ← linear space")
print("channel lam[-1,1]:", lam[-1, 1], "  ← λ = exp(0.5·a)")

In [ ]:
def make_nuts_fn(ssp_priors, y, Z, positivity_idx):
    a0                  = ssp_priors["a0"]
    P0                  = ssp_priors["P0"]
    sigma_q_loc_prior   = ssp_priors["sigma_q_loc_prior"]
    sigma_q_scale_prior = ssp_priors["sigma_q_scale_prior"]
    a_obs               = ssp_priors["a_obs"]
    P_obs               = ssp_priors["P_obs"]
    positivity_mask     = positivity_idx.astype(bool)

    def nuts_fn():
        sigma_h = numpyro.sample(
            "sigma_h",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-5),
        )
        sigma_q_raw = numpyro.sample(
            "sigma_q",
            dist.TruncatedNormal(sigma_q_loc_prior, sigma_q_scale_prior, high=0.1, low=1e-5),
        )
        n_regressors = Z.shape[-1] - 1
        sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

        lp, at, Pt, vt, Ft, Kt = kalman_filter_1d_ekf(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            positivity_idx=positivity_mask,
            a_obs=a_obs,
            P_obs=P_obs,
        )
        lam = jnp.where(positivity_mask, jnp.exp(jnp.clip(0.5 * at, -10.0, 10.0)), at)

        at_smooth = kalman_dk_smoother_1d_ekf(
            at, Pt, vt, Ft, Kt, Z, a0, P0, sigma_q,
            positivity_idx=positivity_mask,
            a_obs=a_obs,
            P_obs=P_obs,
        )
        lam_smooth = jnp.where(
            positivity_mask, jnp.exp(jnp.clip(0.5 * at_smooth, -10.0, 10.0)), at_smooth
        )

        numpyro.factor("lp", lp)
        numpyro.deterministic("at", at)
        numpyro.deterministic("lam", lam)
        numpyro.deterministic("at_smooth", at_smooth)
        numpyro.deterministic("lam_smooth", lam_smooth)
        numpyro.deterministic("mu", jnp.sum(Z * lam, -1))
        numpyro.deterministic("mu_smooth", jnp.sum(Z * lam_smooth, -1))

    return nuts_fn

In [ ]:
nuts_fn = make_nuts_fn(ssp_priors, y, Z, positivity_idx)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()

# detailed per-parameter diagnostics for scalar params
samples = mcmc.get_samples(group_by_chain=True)
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
state_labels = ["intercept"] + regressors
plot_obs_loc = a_to_lam(a_obs, exponent=0.5, positivity_idx=positivity_mask) if a_obs is not None else None
plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="lam",
    coefs_df=coefs_df,
    obs_idx=obs_idx if USE_DISCLOSURE else None,
    a_obs=plot_obs_loc,
    P_obs=P_obs,
    title="Log-normal EKF — λ_t = exp(0.5·a_t)  [disclosure ON]" if USE_DISCLOSURE else "Log-normal EKF — λ_t = exp(0.5·a_t)  [disclosure OFF]",
)
plt.show()

In [ ]:
plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="lam_smooth",
    coefs_df=coefs_df,
    obs_idx=obs_idx if USE_DISCLOSURE else None,
    a_obs=plot_obs_loc,
    P_obs=P_obs,
    title="Log-normal EKF — smoothed λ_t = exp(0.5·a_t)  [disclosure ON]" if USE_DISCLOSURE else "Log-normal EKF — smoothed λ_t = exp(0.5·a_t)  [disclosure OFF]",
)
plt.show()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
eps_samples = np.random.normal(
    0,
    np.array(posteriors_dict["sigma_h"])[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)
# yhat_lower, yhat_mid, yhat_upper = np.quantile((mu_samples + eps_samples) * response_std + response_mean, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')